# Capirotada - PEX (interactivo)

Capsula 12, el cierre. El layout agrega **capacitancia parasita** que el
esquematico ignoraba. La relacion clave del ancho de banda:

$$GBW = \frac{g_m}{2\pi\,(C_L + C_{par})}$$

Mas C parasita -> menos GBW. Aqui movemos `Cpar` y vemos el GBW caer -- la
misma pregunta de signoff: **¿aun cumple?**

> Valores del **OTA real** de este evento (del `.ac` de ngspice):
> Av0=40.8 dB, GBW=17.4 MHz, CL=1 pF. La ultima seccion superpone tus Bodes
> **reales** (ideal `bode.txt` + con PEX `bode_pex.txt`) si los subes.

Corre las celdas en orden.

## 1 - El modelo (Cpar baja el GBW)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

F = np.logspace(0, 9, 600)
GM = 109e-6; RO = 1.0e6; CL = 1e-12; FP2 = 100e6   # ~OTA real (Av0=40.8dB, GBW=17.4MHz)

def bode(Cpar_fF):
    Ctot = CL + Cpar_fF*1e-15
    Av0 = GM*RO; GBW = GM/(2*np.pi*Ctot); fp1 = GBW/Av0
    H = Av0/((1+1j*F/fp1)*(1+1j*F/FP2))
    return 20*np.log10(np.abs(H)), np.angle(H, deg=True), Av0, GBW

def cruce0(f, g):
    for i in range(len(g)-1):
        if g[i] >= 0 >= g[i+1]:
            return 10**(np.log10(f[i]) + (0-g[i])*(np.log10(f[i+1])-np.log10(f[i]))/(g[i+1]-g[i]))
    return None
print('modelo listo (OTA real de Capirotada)')

## 2 - Mueve la capacitancia parasita

`Cpar=0` es el **ideal** (esquematico). Sube `Cpar` y mira el GBW bajar. La
extraccion PEX real de nuestro OTA dio un GBW ~2% menor -> equivale a unos
**~23 fF** de C parasita efectiva (layout chiquito, polo dominado por CL).

In [ ]:
def explora(Cpar_fF=0.0):
    g, p, Av0, GBW = bode(Cpar_fF)
    g0, _, _, GBW0 = bode(0.0)          # referencia ideal
    fu = cruce0(F, g); fu0 = cruce0(F, g0)
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.5, 6), sharex=True)
    a1.semilogx(F, g0, '--', color='#9aa0a6', lw=1.5, label='ideal (Cpar=0)')
    a1.semilogx(F, g,  color='#C8742B', lw=2, label=f'Cpar={Cpar_fF:.0f} fF')
    a1.axhline(0, color='#888', lw=0.8, ls='--')
    if fu: a1.plot([fu],[0],'o',color='#dc2626')
    dd = 100*(GBW-GBW0)/GBW0
    a1.set_title(f'Av0={20*np.log10(Av0):.1f} dB   GBW={GBW/1e6:.2f} MHz   (vs ideal {dd:+.1f}%)')
    a1.set_ylabel('Ganancia [dB]'); a1.grid(True, which='both', alpha=0.3); a1.legend(fontsize=8)
    a2.semilogx(F, p, color='#C8742B', lw=2)
    a2.set_ylabel('Fase [deg]'); a2.set_xlabel('Frecuencia [Hz]'); a2.grid(True, which='both', alpha=0.3)
    plt.tight_layout(); plt.show()

interact(explora, Cpar_fF=FloatSlider(value=0, min=0, max=500, step=10,
         description='Cpar (fF)'));

## 3 - (Opcional) Tus Bodes reales de ngspice

Sube a Colab el `bode.txt` (ideal, de `10-simulacion/results/`) y el
`bode_pex.txt` (con parasitos, de `12-pex/results/`) y corre la celda: dibuja
las curvas **reales** de SKY130, ideal vs PEX.

In [ ]:
import os
def carga(path):
    d = np.atleast_2d(np.loadtxt(path)); o = np.argsort(d[:,0])
    return d[o,0], d[o,1], d[o,-1]

ideal = next((c for c in ['bode.txt','bode_tt.txt'] if os.path.exists(c)), None)
pex   = 'bode_pex.txt' if os.path.exists('bode_pex.txt') else None
if not ideal and not pex:
    print('Sube bode.txt y/o bode_pex.txt a Colab y vuelve a correr esta celda.')
else:
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.5, 6), sharex=True)
    for path, col, lab, ls in [(ideal,'#9aa0a6','ideal (real)','--'), (pex,'#C8742B','con PEX (real)','-')]:
        if not path: continue
        f, g, ph = carga(path)
        a1.semilogx(f, g, ls, color=col, lw=2, label=lab)
        a2.semilogx(f, ph, ls, color=col, lw=2, label=lab)
        fu = cruce0(f, g)
        print(f'{lab:16}: Av0={g[0]:.1f} dB' + (f'  GBW={fu/1e6:.2f} MHz' if fu else ''))
    a1.axhline(0, color='#888', lw=0.8, ls='--'); a1.set_ylabel('Ganancia [dB]')
    a1.grid(True, which='both', alpha=0.3); a1.legend(fontsize=8); a1.set_title('OTA 5T - Bode real: ideal vs PEX')
    a2.set_ylabel('Fase [deg]'); a2.set_xlabel('Frecuencia [Hz]'); a2.grid(True, which='both', alpha=0.3); a2.legend(fontsize=8)
    plt.tight_layout(); plt.show()

## 4 - Fin del camino 🎂

El OTA pasa **DRC + LVS + PEX**: es silicio verificado que **cumple con los
parasitos reales**. Listo para fabricar.

Y con eso recorrimos **todo el camino**, en los dos mundos:

| | Flujo | Signoff |
|---|---|---|
| **Digital** 🍪→🍮 | RTL -> sintesis -> PnR | timing (Cajeta) |
| **Analogico** 🧅→🥖 | esquematico -> sim -> layout | PEX (Capirotada) |

Ese es **el camino de disenar chips.** Gracias por acompaniarnos. 🎉